# Notebook 1 -- Core: from transport to diffusion (~45 min)

This notebook trains an ε-network on a 2D spiral target and generates
samples from it by integrating an ODE (ordinary differential equation) --
a complete, working diffusion model.

Three markers:
- ✏️ marks an exercise
- 📦 marks provided code or context (just read/run)
- ⭐ marks optional extra material

## 0. Setup

Every notebook of this tutorial opens with the same setup pattern.
If you ran Notebook 0's install check in advance, this is all the setup
there is.

In [ ]:
# 📦 Canonical preamble.
import time
from pathlib import Path

import jax
import jax.numpy as jnp
import numpy as np
import flax.nnx as nnx
import optax
import bijx
import matplotlib.pyplot as plt
from tqdm import tqdm

import iaifi_gm as gm  # helper package

rngs = nnx.Rngs(0)

In [ ]:
# 📦 Sanity cell: where are we running, and did Notebook 0 stick?
gm.plotting.use_style()
gm.plotting.device_report()

# jit + grad + vmap smoke test (one line of each).
sq_grad = jax.jit(jax.vmap(jax.grad(lambda x: x**2)))
assert jnp.allclose(sq_grad(jnp.arange(3.0)), 2 * jnp.arange(3.0))
print("jit/grad/vmap: ok")

# Repo-relative paths (works from the repo root or from notebooks/).
REPO = Path.cwd() if (Path.cwd() / "checkpoints").exists() else Path.cwd().parent
CKPT_DIR = REPO / "checkpoints"
print(f"checkpoint dir: {CKPT_DIR}  (exists: {CKPT_DIR.exists()})")

## 1. The target & the transport idea (skim, 📦 only)

It demonstrates the transport idea, in four sentences:

1. A generative model is a **map** transporting a simple base density
   (a Gaussian) to the target density.
2. One way to build the map is an ODE $dx/dt = v_\theta(x, t)$ whose
   velocity field $v_\theta$ is a neural network -- a **CNF**
   (continuous normalizing flow).
3. Along a trajectory, the log-density changes by the integral of
   $-\nabla \cdot v$ (the Lagrangian change of variables,
   $\frac{d}{dt} \log p_t(x(t)) = -(\nabla \cdot v)(x(t), t)$) -- so a CNF
   gives **exact log-densities**, not just samples.
4. Computing $\nabla \cdot v$ costs roughly $D$ network passes per point in
   $D$ dimensions -- this cost is what the rest of the notebook removes.

Our target for the whole notebook is the 2D "spiral" from `gm.targets` --
two Gaussian blobs swirled into a pair of curved spiral arms. The cell below trains a
small CNF on it by maximum likelihood using
`bijx`: an MLP (multi-layer perceptron) velocity field wrapped in
`bijx.AutoJacVF` (which adds the divergence via autodiff Jacobians) and
integrated with `bijx.ContFlowRK4` (RK4 = classic 4th-order Runge-Kutta).

In [ ]:
# 📦 Build the CNF: prior N(0, I) pushed through an ODE with a learned
# velocity field.


class CNFVectorField(nnx.Module):
    """CNF vector-field body ``(t, x) -> dx/dt``.

    The integrator assumes vector-fields are ``vf(t, x)``; ``gm.models.TimeMLP`` is
    ``model(x, t)``.  The adapter must be an nnx.Module -- a bare lambda would
    hide the parameters from the optimizer.
    """

    def __init__(self, *, rngs: nnx.Rngs):
        self.net = gm.models.TimeMLP(dim=2, hidden=64, depth=2, time_dim=32, rngs=rngs)

    def __call__(self, t, x):
        return self.net(x, t)


def build_cnf(seed: int = 0) -> bijx.Transformed:
    vf = bijx.AutoJacVF(CNFVectorField(rngs=nnx.Rngs(params=seed)))
    flow = bijx.ContFlowRK4(vf, steps=16)
    prior = bijx.IndependentNormal((2,), rngs=nnx.Rngs(sample=seed + 1))
    return bijx.Transformed(prior, flow)


cnf = build_cnf()
x_demo, log_q_demo = cnf.sample((4,))  # samples AND their exact log-density
print("untrained CNF sample:", np.asarray(x_demo[0]), " log q:", float(log_q_demo[0]))

In [ ]:
# 📦 Train the CNF by maximum likelihood (~20-30 s on a laptop CPU) -- already
# using the canonical train-step pattern you will meet again in section 4.
CNF_STEPS, CNF_BATCH = 600, 256

cnf_opt = nnx.Optimizer(
    cnf, optax.adam(optax.cosine_decay_schedule(3e-3, CNF_STEPS)), wrt=nnx.Param
)


@nnx.jit
def cnf_train_step(cnf, optimizer, key):
    x = gm.targets.sample_spiral(key, CNF_BATCH)

    def loss_fn(cnf):  # negative log-likelihood of the data batch
        return -jnp.mean(cnf.log_density(x))

    loss, grads = nnx.value_and_grad(loss_fn)(cnf)
    optimizer.update(grads=grads, model=cnf)
    return loss


cnf_losses = np.full(CNF_STEPS, np.nan)
cnf_keys = jax.random.split(jax.random.key(2), CNF_STEPS)
t0 = time.perf_counter()
for i in tqdm(range(CNF_STEPS), desc="CNF train"):
    cnf_losses[i] = cnf_train_step(cnf, cnf_opt, cnf_keys[i])
print(f"wall-clock {time.perf_counter() - t0:.1f} s")

final_nll = cnf_losses[-50:].mean()
print(f"final NLL {final_nll:.3f}  (target differential entropy ~ 2.4)")

# Fallback: if the live training misbehaved, load the shipped checkpoint.
CNF_CKPT = CKPT_DIR / "spiral_cnf.msgpack"
if not (np.isfinite(final_nll) and final_nll < 3.0):
    if CNF_CKPT.exists():
        cnf = gm.checkpoints.load(build_cnf(), CNF_CKPT)
        print("Live training misbehaved -- loaded the pre-trained fallback checkpoint.")
    else:
        print("Live training misbehaved and no fallback checkpoint found -- "
              "plots below may look off; this does not affect sections 2-5.")

In [ ]:
# 📦 The payoff: samples AND an exact density, from one model.
x_cnf, _ = cnf.sample((2048,))
x_target = gm.targets.sample_spiral(jax.random.key(0), 2048)


def cnf_log_density(pts):
    flat = pts.reshape(-1, 2)  # flatten the grid to a plain batch
    return cnf.log_density(flat).reshape(pts.shape[:-1])


fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
gm.plotting.scatter2d(x_target, ax=axes[0])
axes[0].set_title("target (spiral)")
gm.plotting.scatter2d(x_cnf, ax=axes[1])
axes[1].set_title("CNF samples")
gm.plotting.density2d(cnf_log_density, ax=axes[2], n=100)
axes[2].set_title("CNF exact density  $q_\\theta(x)$")
fig.tight_layout()

Keep one fact from this figure in mind: the model has an **exact
log-density**.  We are about to give that up, and only track 2b buys it
back.

The catch is cost: every log-density (and thus every training step here)
needs a full ODE solve, with the $\sim D$-network-pass divergence from
above inside it.  In 2D that is cheap; for images ($D \sim 10^3$) it is
not.  The optional cell below measures this cost.

In [ ]:
# ⭐ Optional timing: divergence cost vs dimension (the "why not CNFs" demo).
def _mean_ms(f, x, reps=5):
    jax.block_until_ready(f(x))  # compile
    t0 = time.perf_counter()
    for _ in range(reps):
        jax.block_until_ready(f(x))
    return (time.perf_counter() - t0) / reps * 1e3


for dim in (2, 16, 64, 128):
    net = gm.models.TimeMLP(dim=dim, hidden=64, depth=2, rngs=nnx.Rngs(params=0))
    fwd = jax.jit(lambda x, net=net: net(x, 0.5))

    def fwd_and_div(x, net=net):  # divergence = trace of the Jacobian
        jac = jax.vmap(jax.jacfwd(lambda xi: net(xi, 0.5)))(x)
        return net(x, 0.5), jnp.trace(jac, axis1=-2, axis2=-1)

    x = jnp.zeros((256, dim))
    t_fwd, t_div = _mean_ms(fwd, x), _mean_ms(jax.jit(fwd_and_div), x)
    print(
        f"D = {dim:3d}:  forward {t_fwd:6.2f} ms   "
        f"forward + divergence {t_div:7.2f} ms   ratio {t_div / t_fwd:5.1f}x"
    )
print("(wall-clock understates the ~D FLOP cost at these tiny sizes -- the trend is what matters)")

## 2. Forward noising process

**Convention (all notebooks, the post, and every figure):** $t$ runs from
0 (pure noise) to 1 (data), the direction of generation.  Note that many
papers in the DDPM (denoising diffusion probabilistic models) tradition
run time the other way, with noising forward, so their $t$ is our $1 - t$.
If you have seen diffusion presented as a *noising process* that you can
run indefinitely (an Ornstein-Uhlenbeck flow toward its stationary
Gaussian) note that our $t$ compactifies that infinite half-line into
$(0, 1]$, with the Gaussian end point at $t = 0$.  We parametrize by
generation time because the generative ODE is the only equation we
integrate; the noising direction is sampled in closed form.

Instead of learning a transport map directly, diffusion models **fix the
path** between noise and data by construction.  Take a data sample $x$ and
noise $z \sim N(0, I)$, and define the noising path

$$ x_t = \alpha_t\, x + \sigma_t\, z, $$

with $\alpha_0 = 0,\ \sigma_0 = 1$ (pure noise at $t=0$) and
$\alpha_1 = 1,\ \sigma_1 \approx 0$ (data at $t=1$).  Everything that
follows (loss and sampler) is derived from this.

Our schedule choice here is variance-preserving (meaning
$\alpha_t^2 + \sigma_t^2 = 1$):

$$ \alpha_t = \sin(\pi t / 2), \qquad \sigma_t = \cos(\pi t / 2). $$

One technical issue is worth making explicit: the endpoints are singular --
the score diverges as $\sigma_t \to 0$ (the data end), and the PF-ODE
coefficients blow up as $\alpha_t \to 0$ (the noise end). We therefore clip
$t \in [t_{\min}, 1 - t_{\min}]$ with $t_{\min} = 10^{-2}$ everywhere,
both when sampling $t$ for training and in the sampler grids.  Note that flow
matching (2a) has a better-behaved velocity parametrization.

In [ ]:
# 📦 Schedule (VP trig form) + endpoint clip
T_MIN = 1e-2

alpha = lambda t: jnp.sin(jnp.pi * t / 2)  # alpha_0 = 0 (noise), alpha_1 = 1 (data)
sigma = lambda t: jnp.cos(jnp.pi * t / 2)  # sigma_0 = 1,          sigma_1 = 0

t_plot = jnp.linspace(0, 1, 200)
fig, ax = plt.subplots(figsize=(4.5, 3))
ax.plot(t_plot, alpha(t_plot), label=r"$\alpha_t$ (data weight)")
ax.plot(t_plot, sigma(t_plot), label=r"$\sigma_t$ (noise weight)")
ax.set_xlabel("t   (0 = noise, 1 = data)")
ax.legend()
fig.tight_layout()

### ✏️ Exercise 1: sample the noising path

Implement $x_t = \alpha_t x + \sigma_t z$ with $z \sim N(0, I)$, returning
both $x_t$ and the noise $z$ (the loss in section 3 needs $z$ as its
regression target).

Be careful with shapes: `alpha(t)` has shape `(B,)` but `x` has shape `(B, 2)`.

In [ ]:
def sample_xt(key_eps, x, t):
    """Sample the noising path at times t.

    Args:
        key_eps: PRNG key for the noise draw (pre-split by the caller).
        x: data batch, shape (B, 2).
        t: times in [0, 1], shape (B,).

    Returns:
        (x_t, z): noised batch and the noise used, both shape (B, 2).
    """
    raise NotImplementedError  # YOUR CODE HERE

In [ ]:
# 📦 Test cell for Exercise 1.
B_test = 512
x_test = gm.targets.sample_spiral(jax.random.key(7), B_test)
# PRNG recap (nb0 §2.3): jax.random.split yields fresh, independent keys;
# the same key always reproduces the same draw.
key_eps_test, _ = jax.random.split(jax.random.key(42))

x_t, z = sample_xt(key_eps_test, x_test, jnp.full((B_test,), 0.5))
assert x_t.shape == (B_test, 2), f"x_t has shape {x_t.shape}, expected {(B_test, 2)}"
assert z.shape == (B_test, 2), f"z has shape {z.shape}, expected {(B_test, 2)}"

# Determinism: same key -> same numbers.
x_t2, _ = sample_xt(key_eps_test, x_test, jnp.full((B_test,), 0.5))
assert jnp.allclose(x_t, x_t2), "same key must give the same draw"

# t near 1 (data end): x_t must be close to x.
x_t, _ = sample_xt(key_eps_test, x_test, jnp.full((B_test,), 1 - T_MIN))
assert jnp.max(jnp.abs(x_t - x_test)) < 0.15, (
    "at t = 1 - t_min, x_t should be nearly x -- check which end got alpha"
)

# t near 0 (noise end): x_t must look like N(0, I).
x_t, _ = sample_xt(key_eps_test, x_test, jnp.full((B_test,), T_MIN))
assert abs(float(x_t.mean())) < 0.15 and 0.9 < float(x_t.std()) < 1.1, (
    "at t = t_min, x_t should be (nearly) standard normal"
)
print("Exercise 1 passed.")

In [ ]:
# 📦 The probability path: the spiral dissolving into noise (left = noise
# end, right = data end, the direction of generation).
ts_show = [T_MIN, 0.25, 0.5, 0.75, 1 - T_MIN]
x_path = gm.targets.sample_spiral(jax.random.key(8), 2048)
fig, axes = plt.subplots(1, len(ts_show), figsize=(3 * len(ts_show), 3.2))
for ax, t_val in zip(axes, ts_show):
    x_t, _ = sample_xt(jax.random.key(9), x_path, jnp.full((2048,), t_val))
    gm.plotting.scatter2d(x_t, ax=ax)
    ax.set_title(f"t = {t_val:.2f}")
fig.tight_layout()

## 3. The loss

**Why this is cheap:** the CNF in section 1 needed a full ODE solve plus a
divergence for *every* loss evaluation.  Here the path $p_t$ is fixed by
construction, so training uses single $(x_t, t)$ points with no
simulation.  The network is trained to undo the noise pointwise.

We train a noise-prediction network $\varepsilon_\theta(x_t, t)$ with the
MSE (mean squared error) objective

$$ \mathcal{L}(\theta) = \mathbb{E}_{x, z, t}\!\left[\, w(t)\,
   \lVert \varepsilon_\theta(x_t, t) - z \rVert^2 \,\right],
   \qquad w(t) = 1 \text{ for us,} $$

with $x \sim p_{\text{data}}$, $z \sim N(0, I)$,
$t \sim U[t_{\min}, 1 - t_{\min}]$, and $x_t$ from Exercise 1.

**Why predicting noise is the same as learning the score.**  Define the
**score** $s(x, t) := \nabla_x \log p_t(x)$ -- the gradient of the log of
the noised data density.  Three precise statements connect it to our loss:

- Conditionally on $x$, the noised density is Gaussian, so the
  *conditional* score is closed-form:
  $\nabla_x \log p_t(x_t \mid x) = -z / \sigma_t$.
- The minimizer of an MSE is a conditional expectation:
  $\varepsilon_\theta^*(x, t) = \mathbb{E}[z \mid x_t = x]$.
- Together these give
  $s_{\theta}^*(x,t) = -\varepsilon_\theta^*(x, t)/\sigma_t = \nabla_x \log p_t(x)$
  -- the trained network knows the *marginal* score, even though each
  training example only ever showed it one noisy point and its noise.

In [ ]:
# 📦 The network: a small MLP with a sinusoidal time embedding, called as
# model(x, t).  Provided by gm.models (src/iaifi_gm/models.py).
model = gm.models.TimeMLP(dim=2, hidden=128, depth=3, time_dim=32, rngs=nnx.Rngs(params=0))
n_params = sum(p.size for p in jax.tree.leaves(nnx.state(model, nnx.Param)))
print(f"TimeMLP: {n_params:,} parameters")
print("eps(x, t) shape:", model(x_test[:4], jnp.full((4,), 0.5)).shape)

### ✏️ Exercise 2: the ε-matching loss

Fill in `diffusion_loss`: draw $t \sim U[t_{\min}, 1 - t_{\min}]$ (one per
sample, using `key_t`), build $(x_t, z)$ with `sample_xt` (Exercise 1,
using `key_eps`), and return the mean squared error between
$\varepsilon_\theta(x_t, t)$ and $z$.

In [ ]:
def diffusion_loss(model, key_t, key_eps, x):
    """Epsilon-matching loss.

    Args:
        model: eps-network, called as model(x_t, t).
        key_t, key_eps: pre-split PRNG keys for t and the noise.
        x: data batch, shape (B, 2).

    Returns:
        Scalar loss.
    """
    raise NotImplementedError  # YOUR CODE HERE

In [ ]:
# 📦 Test cell for Exercise 2.
key_t_test, key_eps_test = jax.random.split(jax.random.key(11))
loss_test = diffusion_loss(model, key_t_test, key_eps_test, x_test)
assert jnp.shape(loss_test) == (), f"loss must be a scalar, got shape {jnp.shape(loss_test)}"
assert jnp.isfinite(loss_test), "loss is not finite"
assert 0.1 < float(loss_test) < 10.0, (
    f"untrained loss should be O(1) (E[|z|^2] per dim = 1), got {float(loss_test):.3f}"
)
print(f"Exercise 2 passed -- untrained loss {float(loss_test):.3f} (should be ~ 1).")

## 4. Training

The cell below is the canonical train-step pattern of this tutorial.

### ✏️ Exercise 3: wire up the train step *(optional)*

Fill in (i) the `loss_fn` body (call `diffusion_loss` with the pre-split
keys and the batch) and (ii) the optimizer-update line.

In [ ]:
BATCH = 256
optimizer = nnx.Optimizer(model, optax.adam(1e-3), wrt=nnx.Param)


@nnx.jit
def train_step(model, optimizer, key):
    key_x, key_t, key_eps = jax.random.split(key, 3)
    x = gm.targets.sample_spiral(key_x, BATCH)

    def loss_fn(model):
        """model -> scalar loss on a fresh spiral batch."""
        raise NotImplementedError  # YOUR CODE HERE

    loss, grads = nnx.value_and_grad(loss_fn)(model)
    # Apply the gradients (one line; mutates model in place).
    raise NotImplementedError  # YOUR CODE HERE
    return loss

In [ ]:
# 📦 Escape: if you skipped (or broke) Exercise 3, this cell defines
# the same completed train_step for you, since we need it to continue.
def _provided_train_step():
    @nnx.jit
    def train_step(model, optimizer, key):
        key_x, key_t, key_eps = jax.random.split(key, 3)
        x = gm.targets.sample_spiral(key_x, BATCH)

        def loss_fn(model):
            return diffusion_loss(model, key_t, key_eps, x)

        loss, grads = nnx.value_and_grad(loss_fn)(model)
        optimizer.update(grads=grads, model=model)
        return loss

    return train_step


try:
    train_step(model, optimizer, jax.random.key(0))
    print("Your Exercise 3 train_step works -- using it.")
except NotImplementedError:
    train_step = _provided_train_step()
    print("Using the provided train_step (Exercise 3 skipped).")
except Exception as e:
    train_step = _provided_train_step()
    print(f"Your train_step failed ({e!r}) -- using the provided one.")

# Sanity check: ~50 steps should already pull the loss below its ~1.0 start.
_smoke = np.array([float(train_step(model, optimizer, k))
                   for k in jax.random.split(jax.random.key(2), 50)])
if _smoke[-10:].mean() >= _smoke[:10].mean():
    print(f"Warning: loss did not decrease over 50 steps "
          f"({_smoke[:10].mean():.3f} -> {_smoke[-10:].mean():.3f}) -- "
          f"check the train_step in use.")

In [ ]:
# 📦 Train (~5 s on a fast laptop, under a minute on most machines).
# Fresh init so everyone starts from the same parameters.
# One free upgrade over the setup above: cosine-decaying the learning
# rate to zero noticeably sharpens the final samples at no extra cost.
N_TRAIN_STEPS = 4000

model = gm.models.TimeMLP(dim=2, hidden=128, depth=3, time_dim=32, rngs=nnx.Rngs(params=0))
optimizer = nnx.Optimizer(
    model, optax.adam(optax.cosine_decay_schedule(1e-3, N_TRAIN_STEPS)), wrt=nnx.Param
)

losses = np.full(N_TRAIN_STEPS, np.nan)
train_keys = jax.random.split(jax.random.key(1), N_TRAIN_STEPS)
t0 = time.perf_counter()
for i in tqdm(range(N_TRAIN_STEPS), desc="train"):
    losses[i] = train_step(model, optimizer, train_keys[i])
print(f"wall-clock (incl. jit compile): {time.perf_counter() - t0:.1f} s")
print(f"final loss (mean of last 100): {losses[-100:].mean():.3f}")

In [ ]:
# 📦 Loss curve.  The loss does NOT go to zero: the conditional variance of
# z given x_t is irreducible (~ 0.47 for this target + schedule).
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(losses, lw=0.8)
ax.axhline(0.47, ls="--", c="gray", lw=1, label="irreducible ~ 0.47")
ax.set_xlabel("step")
ax.set_ylabel("ε-matching loss")
ax.set_yscale("log")
ax.legend()
fig.tight_layout()

In [ ]:
# 📦 Save your model (optional, one line).
my_ckpt = gm.checkpoints.save(model, CKPT_DIR / "my_spiral_diffusion.msgpack")
print(f"saved {my_ckpt}")
print("reload later with: gm.checkpoints.load(gm.models.TimeMLP(dim=2, hidden=128, "
      "depth=3, time_dim=32, rngs=nnx.Rngs(params=0)), path)")

## 5. Sampling

We generate by integrating an ODE from noise to data.  Differentiating the
noising path of one sample gives its **conditional** velocity
$\dot{x}_t = \dot\alpha_t x + \dot\sigma_t z$; taking the conditional
expectation $\mathbb{E}[\,\cdot \mid x_t\,]$ (and using
$\varepsilon_\theta \approx \mathbb{E}[z \mid x_t]$ from section 3) turns
it into a **marginal** velocity field we can actually evaluate:

$$ v_\theta(x, t) = \frac{\dot\alpha_t}{\alpha_t}\, x
   + \Big( \dot\sigma_t - \sigma_t \frac{\dot\alpha_t}{\alpha_t} \Big)\,
   \varepsilon_\theta(x, t). $$

The ODE $dx/dt = v_\theta(x, t)$ is the **PF-ODE** (probability-flow ODE):
the deterministic ODE whose flow reproduces the same marginals $p_t$ as
the noising process.  We integrate it forward, $t: t_{\min} \to 1 -
t_{\min}$, starting from $x \sim N(0, I)$.

The **Euler update** on a uniform grid $t_k$ with step $\Delta t$ reads

$$ x \leftarrow x + \Delta t \left( a_k\, x + b_k\,
   \varepsilon_\theta(x, t_k) \right), \qquad
   a_k = \frac{\dot\alpha}{\alpha}\Big|_{t_k}, \quad
   b_k = \Big( \dot\sigma - \sigma \frac{\dot\alpha}{\alpha} \Big)\Big|_{t_k}. $$

For the trig schedule these coefficients simplify to

$$ a_k = \frac{\pi}{2} \cot\!\Big(\frac{\pi t_k}{2}\Big), \qquad
   b_k = -\frac{\pi}{2} \Big/ \sin\!\Big(\frac{\pi t_k}{2}\Big). $$

The 📦 cell below precomputes both arrays; the formulas appear
here only so you can see where they come from.  Note that both blow up
like $1/t$ at the noise end. This is why the grid starts at
$t_{\min} = 10^{-2}$, not much closer to zero.

In [ ]:
# 📦 Time grid + schedule coefficient arrays (precomputed).
N_SAMPLE_STEPS = 50

ts = jnp.linspace(T_MIN, 1 - T_MIN, N_SAMPLE_STEPS + 1)  # noise -> data
t_grid, dts = ts[:-1], jnp.diff(ts)
a_grid = (jnp.pi / 2) / jnp.tan(jnp.pi * t_grid / 2)   # alpha_dot / alpha
b_grid = -(jnp.pi / 2) / jnp.sin(jnp.pi * t_grid / 2)  # sigma_dot - sigma * alpha_dot / alpha

### ✏️ Exercise 4: the PF-ODE Euler sampler *(the core of the notebook)*

Fill in the Euler update inside the loop, 2-3 lines: evaluate
$\varepsilon_\theta(x, t_k)$ (by convention, pass $t$ as a `(B,)` array:
`jnp.full(x.shape[0], t_k)`), then apply
$x \leftarrow x + \Delta t\,(a_k x + b_k \varepsilon)$.
Grid, coefficients, initial noise, and the loop shell are provided.

In [ ]:
def sample_pf_ode(model, z, return_traj=False):
    """Generate samples by Euler-integrating the PF-ODE from noise to data.

    Args:
        model: trained eps-network, called as model(x, t).
        z: initial noise, shape (B, 2)  (x at t = t_min).
        return_traj: if True, also return the full trajectory.

    Returns:
        Samples of shape (B, 2); or trajectory (N_SAMPLE_STEPS + 1, B, 2).
    """
    x = z
    traj = [x]
    for k in range(N_SAMPLE_STEPS):
        t_k, dt, a_k, b_k = t_grid[k], dts[k], a_grid[k], b_grid[k]
        raise NotImplementedError  # YOUR CODE HERE
        traj.append(x)
    return jnp.stack(traj) if return_traj else x

In [ ]:
# 📦 Test cell for Exercise 4: shapes, finiteness, and closeness to the
# target in energy distance (a sample-based distance between distributions;
# 0 iff equal, and we print the same-target noise floor for calibration).
z0 = jax.random.normal(jax.random.key(3), (2048, 2))
x_gen = sample_pf_ode(model, z0)
assert x_gen.shape == (2048, 2), f"expected (2048, 2), got {x_gen.shape}"
assert jnp.all(jnp.isfinite(x_gen)), "samples blew up -- check the update signs"

x_ref = gm.targets.sample_spiral(jax.random.key(4), 2048)
ed = gm.metrics.energy_distance(x_gen, x_ref)
ed_floor = gm.metrics.energy_distance(gm.targets.sample_spiral(jax.random.key(5), 2048), x_ref)
print(f"energy distance model vs target: {float(ed):.4f}   (noise floor {float(ed_floor):.4f})")
assert float(ed) < 0.05, (
    "samples are far from the target -- check the Euler update (and that §4 "
    "training converged: final loss should sit near 0.47, see the loss-curve cell)"
)
print("Exercise 4 passed.")

In [ ]:
# 📦 Samples vs target, with a few ODE trajectories overlaid.
traj = sample_pf_ode(model, jax.random.normal(jax.random.key(6), (512, 2)), return_traj=True)
fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.5))
gm.plotting.scatter2d(x_ref, ax=axes[0])
axes[0].set_title("target")
gm.plotting.scatter2d(traj[-1], ax=axes[1])
for i in range(12):
    axes[1].plot(traj[:, i, 0], traj[:, i, 1], lw=0.8, alpha=0.7, color="C1")
axes[1].set_title(f"model -- {N_SAMPLE_STEPS}-step PF-ODE Euler")
fig.tight_layout()

nfe = N_SAMPLE_STEPS  # one network call per Euler step
print(f"NFE (number of network forward evaluations per sample): {nfe}")
print("~50 is at the quality floor here; 128 buys nothing visible; 8 visibly degrades.")

Every sample cost about 50 network calls,
and a CNF's exact density cost even more.  Making generation cheap -- a
handful of network calls, or just one -- is what track 2a and especially
track 2c are about.

### ⭐ Stretch: stochastic (ancestral) sampling

The PF-ODE is only one member of a family: adding noise of the right
magnitude while correcting the drift with the score leaves all marginals
$p_t$ unchanged.  With the learned score $s_\theta(x, t) =
-\varepsilon_\theta(x, t)/\sigma_t$ the SDE (stochastic differential
equation)

$$ dx = \Big[\, v_\theta(x, t) + \tfrac{1}{2} g_t^2\, s_\theta(x, t) \Big] dt
   + g_t\, dW, \qquad
   g_t^2 = 2 \sigma_t^2 \Big( \frac{\dot\alpha_t}{\alpha_t}
   - \frac{\dot\sigma_t}{\sigma_t} \Big) $$

is the choice that reproduces the VP reverse process.  We integrate it
with Euler-Maruyama: the Euler step plus a $\sqrt{\Delta t}\, g_t\, \xi$
noise term, with $\xi \sim N(0, I)$ and one fresh key per step.  In
discrete form this is what "DDPM ancestral sampling" refers to.  We stay
with the continuous view, because discrete-β DDPM formulas assume the
discrete chain and do not transfer to arbitrary $(\alpha_t, \sigma_t)$.

Stochastic sampling needs more steps than the ODE -- the fresh noise
injected at each step has to be contracted away again -- so we use a finer
grid here.

In [ ]:
# 📦 Finer grid + coefficients for the SDE (g2 = g_t^2).
N_SDE_STEPS = 200
ts_sde = jnp.linspace(T_MIN, 1 - T_MIN, N_SDE_STEPS + 1)
t_sde, dts_sde = ts_sde[:-1], jnp.diff(ts_sde)
a_sde = (jnp.pi / 2) / jnp.tan(jnp.pi * t_sde / 2)
b_sde = -(jnp.pi / 2) / jnp.sin(jnp.pi * t_sde / 2)
g2_sde = 2 * sigma(t_sde) ** 2 * (
    (jnp.pi / 2) / jnp.tan(jnp.pi * t_sde / 2) + (jnp.pi / 2) * jnp.tan(jnp.pi * t_sde / 2)
)

In [ ]:
# ⭐ ✏️ Euler-Maruyama update: drift = a x + b eps + (g2 / 2) * score with
# score = -eps / sigma(t); then add sqrt(g2 * dt) * fresh normal noise.
def sample_sde(model, key, z):
    """Stochastic sampler; z: (B, 2) -> samples (B, 2)."""
    x = z
    noise_keys = jax.random.split(key, N_SDE_STEPS)  # pre-split: one per step
    for k in range(N_SDE_STEPS):
        t_k, dt = t_sde[k], dts_sde[k]
        raise NotImplementedError  # YOUR CODE HERE
    return x

In [ ]:
# 📦 Side-by-side: deterministic vs stochastic sampler (skips gracefully if
# the stretch exercise is unfilled).
try:
    x_sde = sample_sde(model, jax.random.key(12), jax.random.normal(jax.random.key(13), (2048, 2)))
    assert x_sde.shape == (2048, 2) and jnp.all(jnp.isfinite(x_sde))
    ed_sde = gm.metrics.energy_distance(x_sde, x_ref)
    print(f"energy distance, SDE ({N_SDE_STEPS} steps): {float(ed_sde):.4f}   "
          f"(PF-ODE @ {N_SAMPLE_STEPS}: {float(ed):.4f})")
    fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.5))
    gm.plotting.scatter2d(x_gen, ax=axes[0])
    axes[0].set_title(f"PF-ODE Euler ({N_SAMPLE_STEPS} steps, deterministic)")
    gm.plotting.scatter2d(x_sde, ax=axes[1])
    axes[1].set_title(f"SDE Euler-Maruyama ({N_SDE_STEPS} steps, stochastic)")
    fig.tight_layout()
except NotImplementedError:
    print("Stretch exercise not filled in -- skipping the SDE comparison.")

## 6. ⭐ Scale it up: fashion-MNIST (stretch / take-home)

Nothing conceptual changes for images: the same schedule, the same loss,
and the same Euler sampler -- only the data (28×28 grayscale images) and
the network (a small UNet, a conv net with skip connections, from
`gm.models`) are changed.  Training is a 30k-step GPU job -- not laptop
material -- so a pre-trained checkpoint is provided with the repo
(produced by `scripts/train_fmnist_diffusion.py`) and we only *sample*
here.  Both cells below skip gracefully if you don't have the files.

In [ ]:
# 📦 A look at the data (skips if the ~30 MB dataset is not downloaded).
DATA_DIR = REPO / "data"
if (DATA_DIR / "train-images-idx3-ubyte.gz").exists():
    images, labels = gm.data.load_fashion_mnist(split="train", data_dir=DATA_DIR)
    print(f"{images.shape[0]} images, shape {images.shape[1:]}, range [{images.min():.0f}, {images.max():.0f}]")
    gm.plotting.image_grid(images[:16], nrow=8)
else:
    print("fashion-MNIST not downloaded -- skipping the data preview.")
    print("At home: gm.data.load_fashion_mnist(data_dir=REPO / 'data') downloads it (~30 MB).")

In [ ]:
# 📦 Sample from the pre-trained image diffusion model (skips if the
# checkpoint is missing).  Same PF-ODE Euler as section 5 -- note the only
# change: coefficients broadcast over image axes ([:, None, None, None]
# would be needed for per-sample t; here t is shared, so a and b are scalars).
FMNIST_CKPT = CKPT_DIR / "fmnist_diffusion.msgpack"
if FMNIST_CKPT.exists():
    unet = gm.models.SmallUNet(channels=(32, 64, 128), in_channels=1, time_dim=128,
                               rngs=nnx.Rngs(params=0))
    gm.checkpoints.load(unet, FMNIST_CKPT)

    # Sampling grid starts at 1e-2; the training script clips t at 1e-3 (a
    # superset) -- this matches the training script's sample().
    T_MIN_IMG, STEPS_IMG = 1e-2, 200
    ts_img = jnp.linspace(T_MIN_IMG, 1 - T_MIN_IMG, STEPS_IMG + 1)

    @nnx.jit
    def euler_step_img(model, x, t, dt):
        a = (jnp.pi / 2) / jnp.tan(jnp.pi * t / 2)
        b = -(jnp.pi / 2) / jnp.sin(jnp.pi * t / 2)
        return x + dt * (a * x + b * model(x, jnp.full(x.shape[0], t)))

    x_img = jax.random.normal(jax.random.key(0), (16, 28, 28, 1))
    for t_lo, t_hi in zip(ts_img[:-1], ts_img[1:]):
        x_img = euler_step_img(unet, x_img, t_lo, t_hi - t_lo)
    gm.plotting.image_grid(x_img, nrow=8)
    print(f"NFE per image: {STEPS_IMG} UNet calls -- this is the cost 2c attacks.")
else:
    print(f"No image checkpoint at {FMNIST_CKPT} -- skipping (this section is stretch).")
    print("To create it, run scripts/train_fmnist_diffusion.py (needs a GPU); "
          "nothing below depends on it.")

## Wrap-up

In ODE language, this notebook did one thing: we trained a velocity
field's ε-parametrization on a **fixed** noising path (no simulation, no
divergences, just pointwise regression) and then generated by integrating
the PF-ODE from noise to data.

We left two loose ends open on purpose:

- Sampling cost **~50-128 network calls per sample**.  Few-step and
  one-step generation is where tracks 2a (flow matching) and 2c
  (distillation) go.
- We never used an exact likelihood.  The CNF from section 1 had one, and
  track 2b (lattice φ⁴) makes use of it for a physics problem.